In [ ]:
from typing import Literal, Optional, TypeVar
from dataclasses import dataclass
from math import radians
from pathlib import Path
from enum import IntEnum

import numpy as np
from numpy import ndarray
import torch
from torch import Tensor
from torch.utils.dlpack import from_dlpack as dlpack2tensor
from torch.utils.dlpack import to_dlpack as tensor2dlpack
from torchio import LabelMap, ScalarImage, Subject
import cupy as cp
from cupyx.scipy.ndimage import center_of_mass, binary_dilation
from nibabel.loadsave import load as nib_load
from nibabel.nifti1 import Nifti1Image
from diffdrr.drr import DRR
from diffdrr.visualization import plot_img_and_mask, plot_drr, plot_mask
from matplotlib import pyplot as plt


MM = float | int
Pixel = int
MMPerPixel = float | int

Degree = float | int
DegreePerSec = float | int
Radian = float | int
Angle = TypeVar("Angle", Degree, Radian)
type Rot[Angle] = tuple[Angle, Angle, Angle]

Sec = float | int
Point: tuple[float, float, float]

class LabelID(IntEnum):
    AO = 1
    LV = 2
    # AO = 1
    # LA = 2
    # RA = 3
    # LVM = 4
    # LV = 5
    # RV = 6
    # PV = 7

@dataclass
class CArmGeometry:
    sdd: MM             # Source to detector distance
    sod: MM             # Source to object distance
    height: Pixel       # Height of image
    delx: MMPerPixel    # Pixel size in x direction
    _width: Pixel | None = None         # Width of image, default to height
    _dely: MMPerPixel | None = None     # Pixel size in y direction, default to delx
    x0: MM = 0.0             # detector principal point x-offset
    y0: MM = 0.0             # detector principal point y-offset
    
    def __post_init__(self):
        if self._width is None:
            self._width = self.height
        if self._dely is None:
            self._dely = self.delx
    
    @property
    def width(self) -> Pixel:
        assert self._width is not None
        return self._width
    
    @property
    def dely(self) -> MMPerPixel:
        assert self._dely is not None
        return self._dely


@dataclass
class DrrSetting:
    patch_size: int = 256
    orientation_type: Literal["AP", "PA"]|None = "AP"
    parameterization: str = "euler_angles"  # representation of rotation
    convention: str = "ZXY"                 # rotation axis sequence, internal rotation
    mask_to_channels: bool = True           # project masks to different label

def get_reorientation(
        orientation_type: Optional[Literal["AP", "PA"]] = "AP"
) -> torch.Tensor:
    # Frame-of-reference change
    if orientation_type == "AP":
        # Rotates the C-arm about the x-axis by 90 degrees
        reorient = torch.tensor(
            [
                [1, 0, 0, 0],
                [0, 0, -1, 0],
                [0, 1, 0, 0],
                [0, 0, 0, 1],
            ],
            dtype=torch.float32,
        )
    elif orientation_type == "PA":
        # Rotates the C-arm about the x-axis by 90 degrees
        # Reverses the direction of the y-axis
        reorient = torch.tensor(
            [
                [1, 0, 0, 0],
                [0, 0, 1, 0],
                [0, 1, 0, 0],
                [0, 0, 0, 1],
            ],
            dtype=torch.float32,
        )
    elif orientation_type is None:
        # Identity transform
        reorient = torch.tensor(
            [
                [1, 0, 0, 0],
                [0, 1, 0, 0],
                [0, 0, 1, 0],
                [0, 0, 0, 1],
            ],
            dtype=torch.float32,
        )
    else:
        raise ValueError(f"Unrecognized orientation {orientation_type}")
    return reorient


def recenter(
    original_affine: ndarray,
    center_voxel: tuple[int, int, int]
) -> ndarray:
    """get the affine that set the center of the image to the given center_voxel
    """
    B = original_affine[:3, :3]
    new_t = -(B @ np.array(center_voxel))
    T = np.eye(4)
    T[:3, :3] = B
    T[:3, 3] = new_t
    return T


def get_coronary_centering_affine(label: Tensor, volume_affine: ndarray) -> ndarray:
    """
    Compute an affine transform that recenters the heart region in world coordinates for CTA
    Args:
        label (Tensor): Binary label mask of shape (1, 1, D, H, W).
        volume_affine (ndarray): Original volume affine matrix of shape (4, 4).

    Returns:
        ndarray: New affine matrix that centers the label's structure in world space.
    """
    ref_label = (label > 0) & (label!=LabelID.AO)
    with cp.cuda.Device(label.device.index):
        label_center = center_of_mass(cp.from_dlpack(tensor2dlpack(ref_label.squeeze().to(label.device)))) # type: ignore
        label_center = (int(label_center[0]), int(label_center[1]), int(label_center[2]))
        return recenter(volume_affine, label_center)


def pre_load(file: Path) -> tuple[Nifti1Image, ndarray]:
    if not file.exists():
        raise FileNotFoundError(f"nii file not found: {file}")
    image_nii = nib_load(file)
    assert isinstance(image_nii, Nifti1Image)
    affine = image_nii.affine if image_nii.affine is not None else np.eye(4)
    return image_nii, affine


def load_nifti(file: Path, is_label: bool = False) -> tuple[Tensor, ndarray]:
    """load nifti file as torch tensor (shape: 1, 1, D, H, W) and return its affine matrix"""
    img, affine = pre_load(file)
    tensor = torch.from_numpy(img.get_fdata())
    if is_label:
        tensor = tensor.round().to(torch.uint8)
    else:
        tensor = tensor.to(torch.float32)
    tensor = tensor[None][None]  # add batch and channel dim
    return tensor, affine


def adjust_iodine_contrast(volume: Tensor, label: Tensor) -> Tensor:
    res = volume.clone()
    # CTA造影剂区域恢复正常血液（与水接近，为0）
    threshold_mask = (volume > 150) & (volume < 400)
    res[(label > 0) | threshold_mask ] = 0.
    
    # 体外仪器的线
    res[res > 1500] = res.min()
    
    res[(res > -150) & (res < 0.)] = 0.
    res[res < -150] = res.min()
    
    res = 0.0001 * (res/1000 + 1).clamp(min=0., max=2.)
    res_max = res.max()
    res /= res_max
    res = res ** 2
    res *= res_max
    return res


def drr_projection(
    img_path: Path,
    seg_path: Path,
    whole_heart_label_path: Path,
    alpha: Angle,
    beta: Angle,
    geom: CArmGeometry,
    config: DrrSetting = DrrSetting()
) -> Tensor:
    device = torch.device('cuda')

    volume, volume_affine = load_nifti(img_path)
    label, _ = load_nifti(seg_path, is_label=True)
    whole_heart_label, _ = load_nifti(whole_heart_label_path, is_label=True)
    volume = volume.to(device)
    label = label.to(device)
    whole_heart_label = whole_heart_label.to(device)
    
        
    bone_mask_coords = torch.where((volume > 600).squeeze())
    w, h, d = volume.shape[-3:]
    x_min = bone_mask_coords[0].min()
    x_max = bone_mask_coords[0].max()
    y_min = bone_mask_coords[1].min()
    y_max = bone_mask_coords[1].max()
    z_min = bone_mask_coords[2].min()
    z_max = bone_mask_coords[2].max()
    x_min = max(x_min-20, 0)
    x_max = min(x_max+20, w)
    y_min = max(y_min-20, 0)
    y_max = min(y_max+20, h)
    z_min = max(z_min-20, 0)
    z_max = min(z_max+20, d)
    
    volume = volume[:, :, x_min:x_max, y_min:y_max, z_min:z_max]
    label = label[:, :, x_min:x_max, y_min:y_max, z_min:z_max]
    whole_heart_label = whole_heart_label[:, :, x_min:x_max, y_min:y_max, z_min:z_max]
    
    whole_heart_label[whole_heart_label != 2] = 0
    affine = get_coronary_centering_affine(whole_heart_label, volume_affine)
    volume = adjust_iodine_contrast(volume, label)
    
    assert volume.dim() >= 3
    w, h, d = volume.shape[-3:]
    shape = (1, w, h, d)
    
    subject = Subject(
        volume=ScalarImage(tensor=volume.float().reshape(*shape).to(device), affine=affine),
        mask=LabelMap(tensor=label.reshape(*shape).to(device), affine=affine),
        reorient = get_reorientation("AP"),    # type: ignore
        density = ScalarImage(tensor=volume.float().reshape(*shape).to(device), affine=affine),
        fiducials = None,   #type: ignore
    )
    
    drr = DRR(
        subject     =   subject,
        sdd         =   geom.sdd,
        height      =   geom.height,
        width       =   geom.width,
        delx        =   geom.delx,
        dely        =   geom.dely,
        x0          =   geom.x0,
        y0          =   geom.y0,
        patch_size  =   config.patch_size,
        renderer    =   "trilinear",
        checkpoint_gradients=True
    ).to(device).eval()
    
    rot: Rot[Radian] = (radians(alpha), radians(beta), 0)
    rotations = torch.tensor([rot], device=device)
    translations = torch.tensor([[0.0, geom.sod, 0.0]], device=device)
    
    res: Tensor = drr(
        rotations,
        translations, 
        mask_to_channels = config.mask_to_channels,
        parameterization = "euler_angles", 
        convention       = "ZXY"
    )
    
    return res


In [ ]:
alpha: Degree = -20.4  # -10.4
beta: Degree = -19.6
img_path = Path("data/3D/xu_total.nii.gz")
seg_path = Path("data/3D/xu_total_seg_partial.nii.gz")
whole_heart_label_path = Path("data/3D/xu_whole_heart_seg.nii.gz")
geom = CArmGeometry(
    sdd = 943.0,
    sod = 600,  # 736.9425104
    height = 512,
    delx = 0.368,
)

proj = drr_projection(
    img_path, seg_path, whole_heart_label_path, alpha, beta, geom
)

In [ ]:
fig, axs = plt.subplots(
    nrows=1,
    ncols=2,
    tight_layout=True,
    dpi=300,
    figsize = (5.0, 2.5),
)

im = proj.sum(dim=1, keepdim=True)
im = torch.exp(-im)
plot_drr(im, axs=axs[0], ticks=False)
plot_drr(im, axs=axs[1], ticks=False)
plot_mask(proj[:, [LabelID.AO, LabelID.LV]], axs=axs[1])

In [ ]:
import nibabel as nib
nib.loadsave.save(nib.nifti1.Nifti1Image(im.rot90(dims=(-1,-2)).squeeze()[None].cpu().numpy(), affine=np.eye(4)),"data/im.nii.gz")

In [ ]:
im.squeeze()[None]